# Create MEIs

In [3]:
from neuro_data.static_images import data_schemas as data, stats
from neuro_data.static_images.data_schemas import meso, experiment
import datajoint as dj
from staticnet_experiments import models, configs
from staticnet_analyses import multi_mei, closed_loop
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

dj.config['display.limit'] = 30

In [4]:
gid_restr = dict(group_id=14)

## Compute unit oracles (used later for unit selection)

Restrict to scan and desired data config to compute the Oracle score

In [7]:
dataset = data.StaticMultiDataset & gid_restr
dataconfig = configs.DataConfig.CorrectedAreaLayer() & {'stimulus_type': 'stimulus.Frame', 'exclude': '', 'layer': 'L2/3', 'brain_area': 'V1', 'normalize_per_image': False}
cond = dataset * dataconfig

stats.Oracle.populate(cond.proj())

## Fill TargetDataset

Only fill `TargetDataset` with specified `group_id` and `DataConfig` that already has `Model`s trained on it and has `Oracle` score computed.

In [8]:
dataset = data.StaticMultiDataset & gid_restr
dataconfig = configs.DataConfig.CorrectedAreaLayer() & {'stimulus_type': 'stimulus.Frame', 'exclude': '', 'layer': 'L2/3', 'brain_area': 'V1', 'normalize_per_image': False}
cond = dataset * dataconfig
cond = cond & (configs.NetworkConfig.CorePlusReadout * models.Model) # has trained models
cond = cond & stats.Oracle.Scores # has oracle computed

multi_mei.TargetDataset.populate(cond, display_progress=True)

# multi_mei.TargetDataset().populate(gid_restr, 
#                              configs.DataConfig.CorrectedAreaLayer & dict(stimulus_type='stimulus.Frame', exclude='', normalize_per_image=False),
#                              configs.NetworkConfig.CorePlusReadout * models.Model,
#                              stats.Oracle.UnitScores, display_progress=True)

0it [00:00, ?it/s]


## Fill TargetModel with desired models

Fill `TargetModel` table with the best performing CNN and best performing Linear model (part of `closed_loop.candidates`), being sure to only look among `model_candidates` for the experiment.

In [10]:
dataset = data.StaticMultiDataset & {'group_id': 24}
dataconfig = configs.DataConfig.CorrectedAreaLayer() & {'stimulus_type': 'stimulus.Frame', 'exclude': '', 'layer': 'L2/3', 'brain_area': 'V1', 'normalize_per_image': False}

for coreconfig in [configs.CoreConfig.StackedLinearGaussianLaplace, configs.CoreConfig.GaussianLaplace]: #closed_loop.candidates:
    all_models = models.Model & dataset & (configs.NetworkConfig.CorePlusReadout & coreconfig & dataconfig) # trained models with right config

    keys, corrs = all_models.fetch('KEY', 'val_corr')
    best_model = keys[np.argmax(corrs)]
    print(models.Model & best_model)
    
    #multi_mei.TargetModel.populate(best_model)
    
# for cand in closed_loop.candidates: # 
#     best_configs, scores = multi_mei.best_model(models.Model & gid_restr & (closed_loop.model_candidates & cand)).fetch('KEY', 'val_corr')
#     print(scores)
#     multi_mei.TargetModel.populate(best_configs)

*group_id    *net_hash      *seed     val_corr     model     
+----------+ +------------+ +-------+ +----------+ +--------+
24           7f11ada69c8fa2 99999     0.283189     =BLOB=    
 (Total: 1)

*group_id    *net_hash      *seed     val_corr     model     
+----------+ +------------+ +-------+ +----------+ +--------+
24           403a616c4761b7 99999     0.319908     =BLOB=    
 (Total: 1)



## Select "best" units to generate MEI

Define the pairing between the best performing CNN and best performing Linear network, which will be subsequently used to define best performing neurons.

In [25]:
multi_mei.ModelGroup().populate(gid_restr)

Rank all units based on their oracle score

In [29]:
multi_mei.OracleRankedUnit.populate(gid_restr)

Selects units that satisfy the following criteria:

1. Top 50% Oracle score.
1. Top 30% fraction Oracle score on CNN **AND** Top 30% fraction Oracle score on Linear.
1. At least 10 microns away from any border.
1. Units that maximize the difference in fraction Oracle score between CNN and Linear model subject to no two of them being closer than 20um in the X, Y, Z plane.

In [10]:
multi_mei.CorrectedHighUnitSelection().populate(gid_restr)

Excluded 76 / 4099 neurons lying within 10 of edge
Overlap = 48.74 %
791 units remaining
After distance check, 309 units remain


## Create MEIS

In [15]:
selected_units = multi_mei.CorrectedHighUnitSelection & gid_restr & 'hu_rank < 300'
multi_mei.MEI.populate(selected_units)

Working on neuron_id=80, readout_key=group014-20457-5-9-0
14-12-2018:09:25:13 INFO     configs.py           122:	 Ignoring input arguments: ""when creating datasets
14-12-2018:09:25:13 INFO     configs.py           103:	 Loading None dataset with tier=None
14-12-2018:09:25:13 INFO     data_schemas.py      827:	 Fetching data for {'net_hash': '6d0271de31cf0a7fdd05e1e79340a471', 'normalize': 1, 'layer': 'L2/3', 'data_hash': 'c18c87ccb6f2b296173514c39d478360', 'group_id': 14, 'seed': 1009, 'readout_key': 'group014-20457-5-9-0', 'data_description': "V1 L2/3 on stimulus.Frame. normalize=True on all (except '')", 'mei_param_id': '150966b4691c643986d738a5f6656594', 'stats_source': 'all', 'neuron_id': 80, 'brain_area': 'V1'}
14-12-2018:09:25:13 INFO     data_schemas.py      837:	 Data will be (images,behavior,pupil_center,responses)
14-12-2018:09:25:13 INFO     data_schemas.py      840:	 Loading dataset group014-20457-5-9-0 --> /external/cache/static20457-5-9-preproc0.h5
14-12-2018:09:25:13 IN

  0%|          | 0/1000 [00:00<?, ?it/s]

finished step 990 in octave 0


100%|██████████| 1000/1000 [00:04<00:00, 207.37it/s]
